# 99N 高速里程点与 PeMS 站点对比可视化

目的：
1. 绘制整条 99N 高速的 Caltrans 官方里程点位（每 0.1 英里）
2. 绘制 District 3 区域内的 PeMS 99N 高速站点
3. 对比观察偏移原因：是 PeMS 里程桩问题还是元数据坐标问题

In [1]:
import pandas as pd
import numpy as np
import os
import glob
from math import radians, sin, cos, sqrt, atan2
import folium
from folium import plugins
import warnings
warnings.filterwarnings('ignore')

# ============== 配置 ==============
# Caltrans 里程点文件（xlsx）
POSTMILE_FILE = "./pems_output.xlsx"  # 修改为你的文件路径

# PeMS 元数据目录
META_DIR = "../d03_meta"

# 目标高速
TARGET_FWY = "99"
TARGET_DIR = "N"

# 输出目录
OUTPUT_DIR = "./output/postmile_comparison"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("配置完成！")

配置完成！


## 1. 加载 Caltrans 里程点数据

In [2]:
# 加载里程点数据
postmile_df = pd.read_excel(POSTMILE_FILE)

print(f"里程点数据:")
print(f"  记录数: {len(postmile_df)}")
print(f"  列名: {list(postmile_df.columns)}")
print(f"\n前10行:")
print(postmile_df.head(10))

print(f"\nAbs PM 范围: {postmile_df['Abs PM'].min():.2f} - {postmile_df['Abs PM'].max():.2f}")
print(f"County 分布:")
print(postmile_df['County'].value_counts())

里程点数据:
  记录数: 4163
  列名: ['District', 'County', 'CA PM', 'Abs PM', 'Length', 'Latitude', 'Longitude']

前10行:
   District County  CA PM  Abs PM  Length   Latitude   Longitude
0         6   Kern  L0.00     0.0     0.1  35.006410 -118.951530
1         6   Kern  L0.10     0.1     0.1  35.008175 -118.952088
2         6   Kern  L0.20     0.2     0.1  35.009939 -118.952653
3         6   Kern  L0.30     0.3     0.1  35.011706 -118.953204
4         6   Kern  L0.40     0.4     0.1  35.013470 -118.953768
5         6   Kern  L0.50     0.5     0.1  35.015234 -118.954333
6         6   Kern  L0.60     0.6     0.1  35.016998 -118.954898
7         6   Kern  L0.70     0.7     0.1  35.018137 -118.955260
8         6   Kern  L0.80     0.8     0.1  35.018886 -118.955489
9         6   Kern   0.08     0.9     0.1  35.020130 -118.955882

Abs PM 范围: 0.00 - 416.20
County 分布:
County
Kern           585
Tulare         539
Butte          457
Sutter         420
San Joaquin    387
Merced         374
Fresno         316

## 2. 加载 PeMS 站点元数据

In [3]:
# 查找 District 3 元数据文件
# 99号高速在 D3 区域有站点
pattern = os.path.join(META_DIR, "d03_text_meta_*.txt")
meta_files = glob.glob(pattern)

if not meta_files:
    raise FileNotFoundError("未找到 D3 元数据文件")

meta_file = sorted(meta_files)[-1]
print(f"使用元数据文件: {meta_file}")

META_COLUMNS = [
    'ID', 'Fwy', 'Dir', 'District', 'County', 'City',
    'State_PM', 'Abs_PM', 'Latitude', 'Longitude', 'Length',
    'Type', 'Lanes', 'Name', 'User_ID_1', 'User_ID_2',
    'User_ID_3', 'User_ID_4'
]

# 加载 D3 元数据
meta_df = pd.read_csv(meta_file, sep='\t', names=META_COLUMNS, header=0, dtype={'ID': str, 'Fwy': str})
print(f"加载 {len(meta_df)} 条记录")

使用元数据文件: ../d03_meta/d03_text_meta_2025_12_30.txt
加载 1903 条记录


In [4]:
# 筛选 99N 高速站点（仅 D3 区域）
pems_99n = meta_df[
    (meta_df['Fwy'] == TARGET_FWY) & 
    (meta_df['Dir'] == TARGET_DIR)
].copy()

# 有效坐标检查
pems_99n = pems_99n[
    (pems_99n['Latitude'].notna()) & 
    (pems_99n['Longitude'].notna()) &
    (pems_99n['Latitude'] > 30) &
    (pems_99n['Latitude'] < 42)
].copy()

pems_99n = pems_99n.sort_values('Abs_PM').reset_index(drop=True)

print(f"99{TARGET_DIR} 高速站点 (D3 区域):")
print(f"  站点数: {len(pems_99n)}")
if len(pems_99n) > 0:
    print(f"  Abs_PM 范围: {pems_99n['Abs_PM'].min():.2f} - {pems_99n['Abs_PM'].max():.2f}")
    print(f"\n类型分布:")
    print(pems_99n['Type'].value_counts())
    print(f"\nCounty 分布:")
    print(pems_99n['County'].value_counts())
else:
    print("  警告: 未找到 99N 站点")

99N 高速站点 (D3 区域):
  站点数: 159
  Abs_PM 范围: 274.66 - 383.25

类型分布:
Type
ML    75
OR    36
HV    28
FR    18
FF     2
Name: count, dtype: int64

County 分布:
County
67     120
7       20
101     19
Name: count, dtype: int64


In [5]:
# 显示站点详情
print("PeMS 99N 站点列表:")
print(pems_99n[['ID', 'Type', 'County', 'Abs_PM', 'Latitude', 'Longitude', 'Name']].to_string())

PeMS 99N 站点列表:
          ID Type  County   Abs_PM   Latitude   Longitude                              Name
0    3414051   ML      67  274.660  38.246693 -121.288519               99NB at Crystal Way
1    3414056   FR      67  274.741  38.247736 -121.289182               99NB to Crystal Way
2     319102   FR      67  275.131  38.252741 -121.292418                              C St
3     319091   ML      67  275.400  38.256213 -121.294616                              A St
4     319092   OR      67  275.400  38.256213 -121.294616                              A St
5    3414066   OR      67  278.135  38.292887 -121.313227           Stockton Blvd 99NB Slip
6    3414064   ML      67  278.243  38.294337 -121.314010              99NB JNO Twin Cities
7    3027042   FR      67  281.669  38.341309 -121.334507                99NB to Dillard Rd
8    3027041   ML      67  281.669  38.341309 -121.334507                99NB at Dillard Rd
9    3027012   OR      67  281.750  38.342412 -121.335012        

## 3. 可视化对比

In [6]:
def create_comparison_map(postmile_df, pems_df, output_path):
    """
    创建里程点与 PeMS 站点对比地图
    
    颜色方案：
    - 里程点：蓝色小点
    - PeMS 站点：
      - ML: 红色
      - HV: 紫色
      - OR: 绿色
      - FR: 橙色
      - FF: 黄色
    """
    # 站点颜色
    station_colors = {
        'ML': '#E53935',    # 红色
        'HV': '#8E24AA',    # 紫色
        'OR': '#43A047',    # 绿色
        'FR': '#FB8C00',    # 橙色
        'FF': '#FDD835',    # 黄色
    }
    
    # 计算地图中心
    all_lats = list(postmile_df['Latitude']) + list(pems_df['Latitude'])
    all_lons = list(postmile_df['Longitude']) + list(pems_df['Longitude'])
    center_lat = np.mean(all_lats)
    center_lon = np.mean(all_lons)
    
    # 创建地图
    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=9,
        tiles='OpenStreetMap'
    )
    
    # 添加里程点（蓝色小点）
    postmile_group = folium.FeatureGroup(name='Caltrans 里程点')
    for _, row in postmile_df.iterrows():
        folium.CircleMarker(
            [row['Latitude'], row['Longitude']],
            radius=3,
            color='#1976D2',
            fill=True,
            fillColor='#1976D2',
            fillOpacity=0.7,
            weight=1,
            popup=f"PM: {row['Abs PM']:.2f}<br>County: {row['County']}",
            tooltip=f"PM {row['Abs PM']:.1f}"
        ).add_to(postmile_group)
    postmile_group.add_to(m)
    
    # 添加 PeMS 站点（彩色大点）
    pems_group = folium.FeatureGroup(name='PeMS 站点')
    for _, row in pems_df.iterrows():
        color = station_colors.get(row['Type'], '#888888')
        
        folium.CircleMarker(
            [row['Latitude'], row['Longitude']],
            radius=8,
            color='white',
            weight=2,
            fill=True,
            fillColor=color,
            fillOpacity=0.9,
            popup=folium.Popup(
                f"<b>{row['ID']}</b><br>"
                f"Type: {row['Type']}<br>"
                f"Abs_PM: {row['Abs_PM']:.3f}<br>"
                f"County: {row['County']}<br>"
                f"Name: {row['Name']}<br>"
                f"Lat: {row['Latitude']:.6f}<br>"
                f"Lon: {row['Longitude']:.6f}",
                max_width=300
            ),
            tooltip=f"{row['ID']} ({row['Type']}) PM={row['Abs_PM']:.2f}"
        ).add_to(pems_group)
    pems_group.add_to(m)
    
    # 添加图层控制
    folium.LayerControl().add_to(m)
    
    # 图例
    legend_html = """
    <div style="position: fixed; bottom: 50px; left: 50px; z-index: 1000; 
                background-color: white; padding: 15px; border: 2px solid #333;
                border-radius: 8px; font-size: 12px; font-family: Arial;">
        <div style="font-weight: bold; margin-bottom: 10px; font-size: 14px;">99N 高速对比</div>
        
        <div style="font-weight: bold; margin-bottom: 5px;">Caltrans 里程点:</div>
        <div><span style="display:inline-block; width:12px; height:12px; background:#1976D2; border-radius:50%; margin-right:5px;"></span>0.1 mi 间隔</div>
        
        <div style="font-weight: bold; margin-top: 10px; margin-bottom: 5px;">PeMS 站点:</div>
        <div><span style="display:inline-block; width:12px; height:12px; background:#E53935; border-radius:50%; margin-right:5px;"></span>ML (主线)</div>
        <div><span style="display:inline-block; width:12px; height:12px; background:#8E24AA; border-radius:50%; margin-right:5px;"></span>HV (HOV)</div>
        <div><span style="display:inline-block; width:12px; height:12px; background:#43A047; border-radius:50%; margin-right:5px;"></span>OR (入口)</div>
        <div><span style="display:inline-block; width:12px; height:12px; background:#FB8C00; border-radius:50%; margin-right:5px;"></span>FR (出口)</div>
        <div><span style="display:inline-block; width:12px; height:12px; background:#FDD835; border-radius:50%; margin-right:5px;"></span>FF (连接器)</div>
        
        <div style="margin-top: 10px; font-size: 10px; color: #666;">
            点击站点查看详情
        </div>
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_html))
    
    m.save(output_path)
    print(f"地图已保存: {output_path}")
    return m


# 创建对比地图
comparison_map = create_comparison_map(
    postmile_df, 
    pems_99n,
    os.path.join(OUTPUT_DIR, '99N_postmile_pems_comparison.html')
)

comparison_map

地图已保存: ./output/postmile_comparison/99N_postmile_pems_comparison.html


## 4. 偏移分析

In [7]:
def calc_distance_miles(lat1, lon1, lat2, lon2):
    """Haversine 距离（英里）"""
    R = 3958.8
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    return R * c


def find_nearest_postmile(pems_station, postmile_df):
    """
    找到距离 PeMS 站点最近的里程点
    
    返回两种匹配：
    1. 按 Abs_PM 匹配（应该在同一位置）
    2. 按坐标匹配（实际最近的点）
    """
    pems_pm = pems_station['Abs_PM']
    pems_lat = pems_station['Latitude']
    pems_lon = pems_station['Longitude']
    
    # 1. 按 PM 匹配
    pm_diff = abs(postmile_df['Abs PM'] - pems_pm)
    pm_match_idx = pm_diff.idxmin()
    pm_match = postmile_df.loc[pm_match_idx]
    pm_match_dist = calc_distance_miles(pems_lat, pems_lon, pm_match['Latitude'], pm_match['Longitude'])
    
    # 2. 按坐标匹配
    distances = postmile_df.apply(
        lambda row: calc_distance_miles(pems_lat, pems_lon, row['Latitude'], row['Longitude']),
        axis=1
    )
    coord_match_idx = distances.idxmin()
    coord_match = postmile_df.loc[coord_match_idx]
    coord_match_dist = distances[coord_match_idx]
    
    return {
        'pems_id': pems_station['ID'],
        'pems_type': pems_station['Type'],
        'pems_pm': pems_pm,
        'pems_lat': pems_lat,
        'pems_lon': pems_lon,
        
        # PM 匹配结果
        'pm_match_pm': pm_match['Abs PM'],
        'pm_match_lat': pm_match['Latitude'],
        'pm_match_lon': pm_match['Longitude'],
        'pm_match_dist_mi': pm_match_dist,
        
        # 坐标匹配结果
        'coord_match_pm': coord_match['Abs PM'],
        'coord_match_lat': coord_match['Latitude'],
        'coord_match_lon': coord_match['Longitude'],
        'coord_match_dist_mi': coord_match_dist,
        
        # PM 差异
        'pm_difference': coord_match['Abs PM'] - pems_pm,
    }


# 分析每个 PeMS 站点
analysis_results = []
for _, station in pems_99n.iterrows():
    result = find_nearest_postmile(station, postmile_df)
    analysis_results.append(result)

analysis_df = pd.DataFrame(analysis_results)

print("偏移分析结果:")
print(f"\n站点数: {len(analysis_df)}")

偏移分析结果:

站点数: 159


In [8]:
# 显示偏移分析
print("=" * 80)
print("偏移分析详情")
print("=" * 80)
print(f"{'ID':<10} {'Type':<4} {'PeMS_PM':>8} {'PM匹配距离':>12} {'坐标匹配PM':>10} {'坐标匹配距离':>12} {'PM差异':>10}")
print("-" * 80)

for _, row in analysis_df.iterrows():
    print(f"{row['pems_id']:<10} {row['pems_type']:<4} {row['pems_pm']:>8.2f} "
          f"{row['pm_match_dist_mi']*5280:>10.0f} ft {row['coord_match_pm']:>10.2f} "
          f"{row['coord_match_dist_mi']*5280:>10.0f} ft {row['pm_difference']:>+10.2f}")

偏移分析详情
ID         Type  PeMS_PM       PM匹配距离     坐标匹配PM       坐标匹配距离       PM差异
--------------------------------------------------------------------------------
3414051    ML     274.66        210 ft     274.70        210 ft      +0.04
3414056    FR     274.74        215 ft     274.70        215 ft      -0.04
319102     FR     275.13        163 ft     275.10        163 ft      -0.03
319091     ML     275.40          0 ft     275.40          0 ft      +0.00
319092     OR     275.40          0 ft     275.40          0 ft      +0.00
3414066    OR     278.13        186 ft     278.10        186 ft      -0.03
3414064    ML     278.24        229 ft     278.20        229 ft      -0.04
3027042    FR     281.67        164 ft     281.70        164 ft      +0.03
3027041    ML     281.67        164 ft     281.70        164 ft      +0.03
3027012    OR     281.75        264 ft     281.80        264 ft      +0.05
3027011    ML     281.76        232 ft     281.80        232 ft      +0.04
3032111    ML 

In [9]:
# 统计分析
print("\n" + "=" * 60)
print("偏移统计")
print("=" * 60)

# PM 匹配距离统计（按 PM 值匹配后的坐标偏移）
print("\n1. 按 PM 值匹配后的坐标偏移（英里）:")
print(f"   平均: {analysis_df['pm_match_dist_mi'].mean():.4f} mi ({analysis_df['pm_match_dist_mi'].mean()*5280:.0f} ft)")
print(f"   最大: {analysis_df['pm_match_dist_mi'].max():.4f} mi ({analysis_df['pm_match_dist_mi'].max()*5280:.0f} ft)")
print(f"   最小: {analysis_df['pm_match_dist_mi'].min():.4f} mi ({analysis_df['pm_match_dist_mi'].min()*5280:.0f} ft)")

# 坐标匹配距离统计（坐标最近点的偏移）
print("\n2. 按坐标匹配后的距离（英里）:")
print(f"   平均: {analysis_df['coord_match_dist_mi'].mean():.4f} mi ({analysis_df['coord_match_dist_mi'].mean()*5280:.0f} ft)")
print(f"   最大: {analysis_df['coord_match_dist_mi'].max():.4f} mi ({analysis_df['coord_match_dist_mi'].max()*5280:.0f} ft)")
print(f"   最小: {analysis_df['coord_match_dist_mi'].min():.4f} mi ({analysis_df['coord_match_dist_mi'].min()*5280:.0f} ft)")

# PM 差异统计
print("\n3. PM 差异（坐标最近点 PM - PeMS PM）:")
print(f"   平均: {analysis_df['pm_difference'].mean():+.3f} mi")
print(f"   最大: {analysis_df['pm_difference'].max():+.3f} mi")
print(f"   最小: {analysis_df['pm_difference'].min():+.3f} mi")
print(f"   标准差: {analysis_df['pm_difference'].std():.3f} mi")


偏移统计

1. 按 PM 值匹配后的坐标偏移（英里）:
   平均: 0.0255 mi (134 ft)
   最大: 0.0523 mi (276 ft)
   最小: 0.0000 mi (0 ft)

2. 按坐标匹配后的距离（英里）:
   平均: 0.0255 mi (134 ft)
   最大: 0.0523 mi (276 ft)
   最小: 0.0000 mi (0 ft)

3. PM 差异（坐标最近点 PM - PeMS PM）:
   平均: +0.007 mi
   最大: +0.050 mi
   最小: -0.046 mi
   标准差: 0.028 mi


In [10]:
# 判断偏移原因
print("\n" + "=" * 60)
print("偏移原因分析")
print("=" * 60)

# 分类
for _, row in analysis_df.iterrows():
    pm_dist_ft = row['pm_match_dist_mi'] * 5280
    coord_dist_ft = row['coord_match_dist_mi'] * 5280
    pm_diff = row['pm_difference']
    
    if pm_dist_ft < 100:  # PM匹配距离 < 100 ft
        reason = "✓ 正常（PM和坐标都匹配）"
    elif coord_dist_ft < 100 and abs(pm_diff) > 0.1:
        reason = "⚠️ PeMS PM 值有误（坐标正确但PM不对）"
    elif pm_dist_ft > 500 and coord_dist_ft > 500:
        reason = "❌ PeMS 坐标有误（离高速较远）"
    elif pm_dist_ft > 100 and coord_dist_ft < 100:
        reason = "⚠️ PeMS PM 值有误（坐标正确）"
    else:
        reason = "? 需要进一步检查"
    
    print(f"{row['pems_id']:<10} {row['pems_type']:<4} PM偏移={pm_dist_ft:>6.0f}ft  坐标偏移={coord_dist_ft:>6.0f}ft  {reason}")


偏移原因分析
3414051    ML   PM偏移=   210ft  坐标偏移=   210ft  ? 需要进一步检查
3414056    FR   PM偏移=   215ft  坐标偏移=   215ft  ? 需要进一步检查
319102     FR   PM偏移=   163ft  坐标偏移=   163ft  ? 需要进一步检查
319091     ML   PM偏移=     0ft  坐标偏移=     0ft  ✓ 正常（PM和坐标都匹配）
319092     OR   PM偏移=     0ft  坐标偏移=     0ft  ✓ 正常（PM和坐标都匹配）
3414066    OR   PM偏移=   186ft  坐标偏移=   186ft  ? 需要进一步检查
3414064    ML   PM偏移=   229ft  坐标偏移=   229ft  ? 需要进一步检查
3027042    FR   PM偏移=   164ft  坐标偏移=   164ft  ? 需要进一步检查
3027041    ML   PM偏移=   164ft  坐标偏移=   164ft  ? 需要进一步检查
3027012    OR   PM偏移=   264ft  坐标偏移=   264ft  ? 需要进一步检查
3027011    ML   PM偏移=   232ft  坐标偏移=   232ft  ? 需要进一步检查
3032111    ML   PM偏移=   213ft  坐标偏移=   213ft  ? 需要进一步检查
3027032    ML   PM偏移=   171ft  坐标偏移=   171ft  ? 需要进一步检查
317145     FR   PM偏移=   190ft  坐标偏移=   190ft  ? 需要进一步检查
317146     ML   PM偏移=   142ft  坐标偏移=   142ft  ? 需要进一步检查
317144     OR   PM偏移=    84ft  坐标偏移=    84ft  ✓ 正常（PM和坐标都匹配）
317143     ML   PM偏移=   211ft  坐标偏移=   211ft  ? 需要进一步检查
317142     OR   PM偏移=   2

In [11]:
# 保存分析结果
analysis_df.to_csv(os.path.join(OUTPUT_DIR, '99N_offset_analysis.csv'), index=False)
print(f"\n分析结果已保存到: {OUTPUT_DIR}/99N_offset_analysis.csv")


分析结果已保存到: ./output/postmile_comparison/99N_offset_analysis.csv


## 5. 带连线的详细对比地图

In [12]:
def create_detailed_comparison_map(analysis_df, postmile_df, output_path):
    """
    创建详细对比地图，包含:
    - 里程点
    - PeMS 站点
    - 连线显示偏移
    """
    # 计算中心
    center_lat = analysis_df['pems_lat'].mean()
    center_lon = analysis_df['pems_lon'].mean()
    
    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=9,
        tiles='OpenStreetMap'
    )
    
    station_colors = {
        'ML': '#E53935',
        'HV': '#8E24AA',
        'OR': '#43A047',
        'FR': '#FB8C00',
        'FF': '#FDD835',
    }
    
    # 添加里程点（蓝色连线）
    postmile_sorted = postmile_df.sort_values('Abs PM')
    postmile_coords = [[row['Latitude'], row['Longitude']] for _, row in postmile_sorted.iterrows()]
    
    folium.PolyLine(
        postmile_coords,
        color='#1976D2',
        weight=3,
        opacity=0.7,
        tooltip='Caltrans 里程线'
    ).add_to(m)
    
    # 每隔 1 英里标记一个里程点
    for _, row in postmile_df.iterrows():
        pm = row['Abs PM']
        if pm % 1.0 < 0.05:  # 整数里程
            folium.CircleMarker(
                [row['Latitude'], row['Longitude']],
                radius=5,
                color='#1976D2',
                fill=True,
                fillColor='white',
                fillOpacity=1,
                weight=2,
                popup=f"PM {pm:.0f}",
                tooltip=f"PM {pm:.0f}"
            ).add_to(m)
    
    # 添加 PeMS 站点和偏移连线
    for _, row in analysis_df.iterrows():
        color = station_colors.get(row['pems_type'], '#888888')
        pm_dist_ft = row['pm_match_dist_mi'] * 5280
        
        # PeMS 站点
        folium.CircleMarker(
            [row['pems_lat'], row['pems_lon']],
            radius=10,
            color='white',
            weight=2,
            fill=True,
            fillColor=color,
            fillOpacity=0.9,
            popup=folium.Popup(
                f"<b>PeMS: {row['pems_id']}</b><br>"
                f"Type: {row['pems_type']}<br>"
                f"Abs_PM: {row['pems_pm']:.3f}<br>"
                f"<hr>"
                f"PM匹配偏移: {pm_dist_ft:.0f} ft<br>"
                f"坐标匹配PM: {row['coord_match_pm']:.2f}<br>"
                f"PM差异: {row['pm_difference']:+.2f}",
                max_width=250
            ),
            tooltip=f"{row['pems_id']} ({row['pems_type']}) PM={row['pems_pm']:.2f}"
        ).add_to(m)
        
        # 偏移连线（PeMS → PM匹配点）
        if pm_dist_ft > 50:  # 只显示偏移 > 50 ft 的连线
            line_color = '#F44336' if pm_dist_ft > 500 else '#FFC107'  # 大偏移红色，小偏移黄色
            folium.PolyLine(
                [[row['pems_lat'], row['pems_lon']],
                 [row['pm_match_lat'], row['pm_match_lon']]],
                color=line_color,
                weight=2,
                opacity=0.8,
                dash_array='5,5',
                tooltip=f"偏移: {pm_dist_ft:.0f} ft"
            ).add_to(m)
    
    # 图例
    legend_html = """
    <div style="position: fixed; bottom: 50px; left: 50px; z-index: 1000; 
                background-color: white; padding: 15px; border: 2px solid #333;
                border-radius: 8px; font-size: 12px; font-family: Arial;">
        <div style="font-weight: bold; margin-bottom: 10px;">99N 偏移分析</div>
        
        <div><span style="display:inline-block; width:20px; height:3px; background:#1976D2; margin-right:5px;"></span>Caltrans 里程线</div>
        
        <div style="margin-top: 5px;"><span style="display:inline-block; width:12px; height:12px; background:#E53935; border-radius:50%; margin-right:5px;"></span>ML</div>
        <div><span style="display:inline-block; width:12px; height:12px; background:#8E24AA; border-radius:50%; margin-right:5px;"></span>HV</div>
        <div><span style="display:inline-block; width:12px; height:12px; background:#43A047; border-radius:50%; margin-right:5px;"></span>OR</div>
        <div><span style="display:inline-block; width:12px; height:12px; background:#FB8C00; border-radius:50%; margin-right:5px;"></span>FR</div>
        
        <div style="margin-top: 10px;"><span style="display:inline-block; width:20px; height:2px; background:#FFC107; margin-right:5px; border-style:dashed;"></span>小偏移</div>
        <div><span style="display:inline-block; width:20px; height:2px; background:#F44336; margin-right:5px; border-style:dashed;"></span>大偏移 (>500ft)</div>
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_html))
    
    m.save(output_path)
    print(f"详细对比地图已保存: {output_path}")
    return m


# 创建详细对比地图
detailed_map = create_detailed_comparison_map(
    analysis_df,
    postmile_df,
    os.path.join(OUTPUT_DIR, '99N_detailed_comparison.html')
)

detailed_map

详细对比地图已保存: ./output/postmile_comparison/99N_detailed_comparison.html


## 总结

偏移原因分析：

1. **PM匹配偏移小（< 100 ft）**: PM 值和坐标都正确

2. **PM匹配偏移大，但坐标匹配偏移小**: PeMS 的 Abs_PM 值有误，但 GPS 坐标是正确的

3. **两者偏移都大**: PeMS 的 GPS 坐标本身有误，或者站点不在主线上